In [1]:
import pandas as pd
from pathlib import Path

ROOT      = Path("/Users/emilevandevoorde/Documents/mechelen_parking")
DATA_INT  = ROOT / "data_intermediate"
DATA_PROC = ROOT / "data_processed"

PASS, FAIL = "✓", "⚠️ "
def chk(label, condition, detail=""):
    icon = PASS if condition else FAIL
    print(f"  {icon}  {label}" + (f"  →  {detail}" if detail else ""))

# ── Verwachte bestanden ───────────────────────────────────────────────────────
EXPECTED = {
    DATA_INT  / "shortterm_cleaned.parquet"      : (284524, 21),
    DATA_INT  / "longterm_cleaned.parquet"        : (46643,  20),
    DATA_INT  / "weather_cleaned.parquet"         : (52632,  19),
    DATA_INT  / "calendar_master.parquet"         : (2922,   14),
    DATA_INT  / "parking_location_clean.parquet"  : (11,      7),
    DATA_INT  / "events_master.parquet"           : (2591,   12),
    DATA_PROC / "MAD_shortterm.parquet"           : (284524, None),  # kolommen variëren na events-merge
    DATA_PROC / "MAD_longterm.parquet"            : (46643,  None),
}

print("=" * 55)
print("  BESTANDSCHECK")
print("=" * 55)
all_ok = True
for path, (exp_rows, exp_cols) in EXPECTED.items():
    exists = path.exists()
    chk(f"{path.name:<45} bestaat", exists)
    if not exists:
        all_ok = False
        continue
    df = pd.read_parquet(path)
    row_ok = df.shape[0] == exp_rows
    chk(f"  {'':45} rijen  = {exp_rows:,}", row_ok,
        f"gevonden: {df.shape[0]:,}")
    if exp_cols:
        col_ok = df.shape[1] == exp_cols
        chk(f"  {'':45} kolommen = {exp_cols}", col_ok,
            f"gevonden: {df.shape[1]}")
    if not row_ok:
        all_ok = False

# ── MAD: kritische kolommen aanwezig? ────────────────────────────────────────
print("\n" + "=" * 55)
print("  MAD KOLOMCHECK")
print("=" * 55)

MAD = pd.read_parquet(DATA_PROC / "MAD_shortterm.parquet")

REQUIRED_COLS = {
    "Parkeerdata"  : ["parking_id","rounded_hour","occupancy_rate",
                      "number_of_occupied_spaces","number_of_spaces_override"],
    "Weerdata"     : ["temp_c","precip_mm","wind_speed_ms","humidity_pct"],
    "Kalender"     : ["calendar_day_class","is_national_holiday",
                      "is_school_vacation"],
    "Locatie"      : ["parking_location_category","total_capacity",
                      "longitude","latitude"],
    "Kwal.flags"   : ["low_data_coverage","system_blackout","partial_year"],
    "Events"       : ["is_event_day","is_football_day","is_kermis_day",
                      "is_festival_day","event_scale_max"],
}
for groep, cols in REQUIRED_COLS.items():
    aanwezig = [c for c in cols if c in MAD.columns]
    ontbreekt = [c for c in cols if c not in MAD.columns]
    chk(f"{groep:<14} ({len(aanwezig)}/{len(cols)} kolommen)",
        len(ontbreekt) == 0,
        f"ontbreekt: {ontbreekt}" if ontbreekt else "")

# ── Finale telling ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("  SAMENVATTING")
print("=" * 55)
print(f"  MAD_shortterm kolommen : {MAD.shape[1]}")
print(f"  Trainingsset           : "
      f"{((MAD['low_data_coverage']==0)&(MAD['system_blackout']==0)&(MAD['partial_year']==0)&(MAD['year']>=2020)&(MAD['year']!=2025)).sum():,} rijen")
print(f"  Holdout 2025           : {(MAD['year']==2025).sum():,} rijen")
print(f"  Event-rijen (totaal)   : {MAD['is_event_day'].sum():,}" 
      if "is_event_day" in MAD.columns else "  Events: nog niet gemerged")
print()
print(f"  {'Alle checks geslaagd ✓' if all_ok else 'Eén of meer checks gefaald ⚠️ — zie details hierboven'}")


  BESTANDSCHECK
  ✓  shortterm_cleaned.parquet                     bestaat
  ✓                                                  rijen  = 284,524  →  gevonden: 284,524
  ✓                                                  kolommen = 21  →  gevonden: 21
  ✓  longterm_cleaned.parquet                      bestaat
  ✓                                                  rijen  = 46,643  →  gevonden: 46,643
  ✓                                                  kolommen = 20  →  gevonden: 20
  ✓  weather_cleaned.parquet                       bestaat
  ✓                                                  rijen  = 52,632  →  gevonden: 52,632
  ✓                                                  kolommen = 19  →  gevonden: 19
  ✓  calendar_master.parquet                       bestaat
  ✓                                                  rijen  = 2,922  →  gevonden: 2,922
  ✓                                                  kolommen = 14  →  gevonden: 14
  ✓  parking_location_clean.parquet                b

# Datasamenvatting & onderzoekspositionering
### Algehele databeoordeling

Je staat sterk voor het weers- en kalenderluik van RQ1, maar er zijn twee kritische lacunes die directe aandacht vereisen vóór je verder gaat.

### Wat je sterk maakt

##### Weervariabelen — uitzonderlijk goed
Uurlijkse resolutie, 6+ jaar, volledige tijdreeks zonder temporele gaten, QC-gecorrigeerd met gedifferentieerde strategie per variabele. Dit is beter dan wat de meeste vergelijkbare parkeeronderzoeken gebruiken — de meeste werken met dagelijkse weergemiddelden (Fokker et al., 2021; Tanui et al., 2025). Jij hebt uurlijkse data die direct matcht op de parkeerobservaties.

##### Kalender — volledig en operationeel
Nationale feestdagen, schoolvakanties, calendar_day_class — alle spot-checks correct. Dit is direct bruikbaar als categorische predictor.

##### Tijdsvenster 2025 als holdout
100% dekking, alle 10 parkings, temporeel strikt gescheiden — methodologisch de sterkst mogelijke evaluatiestructuur voor tijdreeksmodellen (Bergmeir & Benítez, 2012; Roberts et al., 2017).

##### Preprocessing — academisch verdedigbaar
Non-destructive flagging (Saunders et al., 2008), MNAR-classificatie van de blackout (Little & Rubin, 2002), fysische plausibiliteitscontrole, validate="many_to_one" op alle joins — elk keuze is traceerbaar en onderbouwd.

### Kritische lacunes voor RQ1
| Element in RQ1                     | Status                 | Impact |
| ---------------------------------- | ---------------------- | ------ |
| Weer                               | ✓ Volledig aanwezig    | —      |
| Kalender                           | ✓ Volledig aanwezig    | —      |
| Evenementen                        | ✗ Afwezig              | Hoog   |
| Prijs                              | ✗ Afwezig              | Hoog   |
| Tier-analyse (centrum/vesten/rand) | ⚠️ Structureel beperkt | Medium |


##### Lacune 1 — Evenementen en prijs ontbreken volledig

RQ1 vraagt expliciet naar "weer, evenementen, kalender, prijs". Momenteel heb je voor twee van de vier variabelengroepen geen data. Dit zijn je opties:

Evenementen: Mechelen heeft een publieke evenementenkalender (stad Mechelen, Uitdatabank). Scrapen of opvragen is haalbaar. Zonder dit kun je enkel zeggen: "evenementen werden niet gemodelleerd wegens databeschikbaarheid" — wat een expliciete scope-inperking vereist in de thesis.

Prijs: Tariefhistoriek van Mechelen Parking is niet publiek beschikbaar. Tenzij je die via stad Mechelen kunt opvragen, moet je prijs uit RQ1 schrappen of herformuleren als "tariefcategorie" (centrum duurder dan rand — als proxy).

##### Lacune 2 — "Rand" is één parking

| Categorie | Parkings                | Totale capaciteit | Analyseerbaar?                   |
| --------- | ----------------------- | ----------------- | -------------------------------- |
| Centrum   | 5                       | 778 pl.           | ✓ Sterk                          |
| Vesten    | 4 (excl. Zandpoortvest) | 787 pl.           | ⚠️ Zandpoortvest ontbreekt (44%) |
| Rand      | 1 (P Keerdok)           | 516 pl.           | ⚠️ Geen generalisatie mogelijk   |

Met één randparking kun je geen uitspraken doen over "rand als categorie" — je kunt enkel zeggen: "P Keerdok vertoont het volgende gedrag". Statistisch gezien is n=1 per definitie niet generaliseerbaar (Field, 2013). Dit is je grootste methodologische kwetsbaarheid voor de tier-analyse.

## Haalbaarheid van RQ1 — component per component

**"Welke externe factoren verbeteren de parkeervoorspelling het meest?"**  
→ **Haalbaar en sterk.** Weer + kalender zijn volledig aanwezig en op uurlijkse resolutie. SHAP-waarden (notebook 12) geven een verdedigbare rangschikking van feature-importantie met correcte onzekerheidsmarge. Zonder evenementen en prijs is de conclusie echter _partieel_ — je meet de bijdrage van de factoren die je hebt, niet van alle relevante factoren.

**"Werken ze anders per parkeertier?"**  
→ **Gedeeltelijk haalbaar.** Centrum vs. vesten is onderbouwd (respectievelijk 5 en 4 parkings met redelijke tijdreeksen). Rand is statistisch niet verdedigbaar als aparte categorie. Aanbeveling: herformuleer de tier-analyse als _centrum vs. niet-centrum_ of behandel P Keerdok als case-study apart.

---

## Aanbevelingen vóór EDA

Twee acties die je onderzoek significant sterker maken zonder grote extra werklast:

**Actie 1 — RQ1 herformuleren (prioriteit: hoog)**  
Scope de onderzoeksvraag expliciet in op de beschikbare variabelen:

> _"Welke **weer- en kalenderfactoren** verbeteren de parkeervoorspelling het meest, en werken ze anders voor **centrumgebonden vs. perifere** parkings in Mechelen?"_

Dit is eerlijker, verdedigbaardere én publiceerbaar zonder de ontbrekende prijs/evenementdata.

**Actie 2 — Evenementendata ophalen (prioriteit: medium)**  
Zelfs een binaire `is_event_day`-kolom van de Mechelen evenementenkalender (Nekker, Hanswijkprocessie, Laundry Day, ...) zou de verklaringskracht van het model aanzienlijk verhogen en je in staat stellen om evenementen toch in RQ1 te includeren.


## Slotbeoordeling

```text
Datakwaliteit (weer, kalender, locatie)  : ████████████░░  8.5/10
Methodologische rigorositeit             : █████████████░  9/10
Dekking van RQ1 met huidige data         : ████████░░░░░░  5.5/10  ← lacunes
Dekking van RQ1 na herformulering        : ████████████░░  8/10
Academische verdedigbaarheid overall     : ████████████░░  8/10

```
De verwerking is methodologisch **sterk en academisch verdedigbaar**. Het centrale risico zit niet in de data-engineering maar in de **scope van RQ1**: die is momenteel breder dan wat je data toelaat. Herformuleer RQ1 nu — vóór de EDA — zodat alle downstream analyses, visualisaties en modelkeuzes coherent zijn met wat je effectief kunt bewijzen.